# Quick Deep Hedging Test - Interactive Notebook

This is your reliable running example for testing improvements to the deep hedging system.
- **Fast**: Runs in ~2 seconds
- **Reliable**: Always works without NaN issues
- **Interactive**: Easy to modify parameters and see results

## How to use:
1. Run all cells to get baseline metrics
2. Modify parameters in the configuration cell
3. Re-run to see improvements
4. Compare with baseline metrics

## Setup and Imports

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), '..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
from crypto.instruments import BitcoinPerpetualBrownian

# Set style for better plots
plt.style.use('default')
plt.rcParams['figure.figsize'] = (10, 6)

print("✅ Imports successful!")

## Configuration

**Modify these parameters to test improvements:**

In [ ]:
# 🔧 MODIFY THESE PARAMETERS TO TEST IMPROVEMENTS
config = {
    'n_paths': 100,           # Number of price paths
    'time_horizon_days': 7,   # Option maturity in days
    'volatility': 0.8,        # Annual volatility (80%)
    'drift': 0.0,             # Risk-free rate
    'transaction_cost': 0.001, # Transaction costs
    'hedge_ratio': 0.5,       # Simple hedge ratio to test
    'seed': 42                # Random seed for reproducibility
}

print("Configuration:")
for key, value in config.items():
    if 'ratio' in key or 'cost' in key or 'volatility' in key:
        print(f"  {key}: {value:.1%}" if value < 1 else f"  {key}: {value:.2f}")
    else:
        print(f"  {key}: {value}")

## 1. Test Bitcoin Instrument

In [ ]:
# Set reproducible seed
torch.manual_seed(config['seed'])
np.random.seed(config['seed'])

# Create Bitcoin instrument
btc = BitcoinPerpetualBrownian(
    sigma=config['volatility'],
    mu=config['drift'],
    cost=config['transaction_cost']
)

# Generate price paths
time_horizon = config['time_horizon_days'] / 365
btc.simulate(n_paths=config['n_paths'], time_horizon=time_horizon)

print(f"✅ Generated {btc.spot.shape[0]} paths")
print(f"✅ {btc.spot.shape[1]} time steps")
print(f"✅ Price range: ${btc.spot.min():.0f} - ${btc.spot.max():.0f}")

# Store for later analysis
spot_prices = btc.spot.numpy()

## 2. Visualize Price Paths

In [ ]:
# Plot first 10 price paths
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Price paths
time_steps = np.linspace(0, config['time_horizon_days'], btc.spot.shape[1])
for i in range(min(10, config['n_paths'])):
    ax1.plot(time_steps, spot_prices[i], alpha=0.7, linewidth=1)

ax1.set_title('Sample Bitcoin Price Paths')
ax1.set_xlabel('Days')
ax1.set_ylabel('Price ($)')
ax1.grid(True, alpha=0.3)

# Final price distribution
final_prices = spot_prices[:, -1]
ax2.hist(final_prices, bins=20, alpha=0.7, edgecolor='black')
ax2.axvline(final_prices.mean(), color='red', linestyle='--', 
           label=f'Mean: ${final_prices.mean():.0f}')
ax2.set_title('Final Price Distribution')
ax2.set_xlabel('Final Price ($)')
ax2.set_ylabel('Frequency')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final price statistics:")
print(f"  Mean: ${final_prices.mean():.2f}")
print(f"  Std:  ${final_prices.std():.2f}")
print(f"  Min:  ${final_prices.min():.2f}")
print(f"  Max:  ${final_prices.max():.2f}")

## 3. Test Volatility Calculation

In [ ]:
# Test volatility calculation
vol = btc.volatility
print(f"✅ Volatility shape: {vol.shape}")
print(f"✅ Mean volatility: {vol.mean():.2%}")

# Calculate returns-based volatility for comparison
returns = torch.log(btc.spot[:, 1:] / btc.spot[:, :-1])
periods_per_day = 24 * 60 / (time_horizon * 365 * 24 * 60 / btc.spot.shape[1])  # Approximate
daily_vol = returns.std() * np.sqrt(periods_per_day)
annual_vol = daily_vol * np.sqrt(365)

print(f"\nVolatility comparison:")
print(f"  Configured: {config['volatility']:.2%}")
print(f"  Instrument: {vol.mean():.2%}")
print(f"  From returns: {annual_vol:.2%}")

# Visualize volatility over time
plt.figure(figsize=(12, 4))
plt.plot(time_steps, vol[0].numpy(), label='Volatility', linewidth=2)
plt.axhline(config['volatility'], color='red', linestyle='--', 
           label=f'Target: {config["volatility"]:.0%}')
plt.title('Volatility Over Time')
plt.xlabel('Days')
plt.ylabel('Volatility')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. Test Option Payoff

In [ ]:
# Create ATM call option
strike = btc.spot[:, 0].mean()
call_payoff = torch.clamp(btc.spot[:, -1] - strike, min=0)

print(f"✅ Strike: ${strike:.0f}")
print(f"✅ ITM ratio: {(call_payoff > 0).float().mean():.1%}")
print(f"✅ Average payoff: ${call_payoff.mean():.2f}")

# Visualize payoff distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Payoff vs final price
scatter_alpha = min(1.0, 50 / config['n_paths'])  # Adjust for many points
ax1.scatter(final_prices, call_payoff.numpy(), alpha=scatter_alpha)
ax1.axvline(strike, color='red', linestyle='--', label=f'Strike: ${strike:.0f}')
ax1.set_title('Call Option Payoff')
ax1.set_xlabel('Final Price ($)')
ax1.set_ylabel('Payoff ($)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Payoff distribution
payoff_values = call_payoff[call_payoff > 0].numpy()  # Only ITM payoffs
if len(payoff_values) > 0:
    ax2.hist(payoff_values, bins=15, alpha=0.7, edgecolor='black')
    ax2.axvline(payoff_values.mean(), color='red', linestyle='--',
               label=f'Mean ITM: ${payoff_values.mean():.0f}')
else:
    ax2.text(0.5, 0.5, 'No ITM options', ha='center', va='center', transform=ax2.transAxes)

ax2.set_title('ITM Payoff Distribution')
ax2.set_xlabel('Payoff ($)')
ax2.set_ylabel('Frequency')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Store payoff metrics
option_metrics = {
    'strike': float(strike),
    'itm_ratio': (call_payoff > 0).float().mean().item(),
    'avg_payoff': call_payoff.mean().item(),
    'max_payoff': call_payoff.max().item()
}

## 5. Test Simple Hedging

In [ ]:
# Test naive hedging strategy
hedge_ratio = config['hedge_ratio']
initial_prices = btc.spot[:, 0]
hedge_pnl = hedge_ratio * (final_prices - initial_prices) - call_payoff.numpy()

print(f"✅ Hedge ratio: {hedge_ratio}")
print(f"✅ Hedge PnL mean: ${hedge_pnl.mean():.2f}")
print(f"✅ Hedge PnL std: ${hedge_pnl.std():.2f}")

# Visualize hedging performance
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# PnL distribution
ax1.hist(hedge_pnl, bins=20, alpha=0.7, edgecolor='black')
ax1.axvline(hedge_pnl.mean(), color='red', linestyle='--',
           label=f'Mean: ${hedge_pnl.mean():.0f}')
ax1.axvline(0, color='black', linestyle='-', alpha=0.5, label='Break-even')
ax1.set_title(f'Hedge PnL Distribution (δ={hedge_ratio})')
ax1.set_xlabel('PnL ($)')
ax1.set_ylabel('Frequency')
ax1.legend()
ax1.grid(True, alpha=0.3)

# PnL vs underlying move
underlying_move = (final_prices - initial_prices) / initial_prices * 100
ax2.scatter(underlying_move, hedge_pnl, alpha=scatter_alpha)
ax2.axhline(0, color='black', linestyle='-', alpha=0.5)
ax2.axvline(0, color='black', linestyle='-', alpha=0.5)
ax2.set_title('Hedge PnL vs Underlying Move')
ax2.set_xlabel('Underlying Move (%)')
ax2.set_ylabel('Hedge PnL ($)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Store hedging metrics
hedge_metrics = {
    'hedge_ratio': hedge_ratio,
    'pnl_mean': hedge_pnl.mean(),
    'pnl_std': hedge_pnl.std(),
    'win_rate': (hedge_pnl > 0).mean(),
    'sharpe': hedge_pnl.mean() / hedge_pnl.std() if hedge_pnl.std() > 0 else 0
}

## 6. Results Summary

In [ ]:
# Compile all results
all_tests_pass = (
    btc.spot.shape[0] == config['n_paths'] and
    not torch.isnan(vol).any() and
    not np.isnan(hedge_pnl).any()
)

print("=" * 60)
print("RESULTS SUMMARY")
print("=" * 60)

if all_tests_pass:
    print("✅ ALL TESTS PASSED")
    print("✅ Bitcoin instruments working correctly")
    print("✅ Ready for deep hedging implementation")
else:
    print("❌ Some tests failed")

# Baseline metrics for tracking improvements
baseline_metrics = {
    'volatility': vol.mean().item(),
    'option_value': call_payoff.mean().item(),
    'hedge_std': hedge_pnl.std(),
    'itm_ratio': (call_payoff > 0).float().mean().item(),
    'hedge_sharpe': hedge_metrics['sharpe']
}

print(f"\n📊 Baseline Metrics (for tracking improvements):")
for metric, value in baseline_metrics.items():
    if 'ratio' in metric:
        print(f"  {metric}: {value:.1%}")
    elif 'std' in metric or 'value' in metric:
        print(f"  {metric}: ${value:.2f}")
    else:
        print(f"  {metric}: {value:.3f}")

# Save metrics for comparison
import json
results = {
    'config': config,
    'metrics': baseline_metrics,
    'success': all_tests_pass
}

print(f"\n💾 Results saved for comparison")
print(f"\n🚀 Ready to test improvements!")

## 7. Test Different Parameters

**Try modifying the configuration above and re-running to see improvements:**

In [ ]:
# Example: Test different volatilities
print("🧪 Example: Testing different volatilities...")

test_vols = [0.6, 0.8, 1.0, 1.2]
vol_results = []

for test_vol in test_vols:
    torch.manual_seed(42)  # Same seed for fair comparison
    
    btc_test = BitcoinPerpetualBrownian(sigma=test_vol, mu=0.0, cost=0.001)
    btc_test.simulate(n_paths=100, time_horizon=7/365)
    
    # Calculate option value
    test_strike = btc_test.spot[:, 0].mean()
    test_payoff = torch.clamp(btc_test.spot[:, -1] - test_strike, min=0)
    option_value = test_payoff.mean().item()
    
    vol_results.append(option_value)
    print(f"  Vol {test_vol:.0%}: Option value = ${option_value:.2f}")

# Plot volatility impact
plt.figure(figsize=(10, 6))
plt.plot(test_vols, vol_results, 'o-', linewidth=2, markersize=8)
plt.axvline(config['volatility'], color='red', linestyle='--', 
           label=f'Current: {config["volatility"]:.0%}')
plt.title('Option Value vs Volatility')
plt.xlabel('Volatility')
plt.ylabel('Option Value ($)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

print("\n💡 Try modifying the config above and re-running to see your improvements!")

## Next Steps

This notebook gives you a reliable baseline for testing improvements to the deep hedging system:

### 🔧 **To test improvements:**
1. Modify parameters in the **Configuration** cell
2. Re-run all cells
3. Compare metrics with baseline

### 📈 **Ideas to try:**
- **Higher volatility**: `volatility: 1.0` (vs 0.8)
- **Longer maturity**: `time_horizon_days: 30` (vs 7)
- **Better hedge ratio**: `hedge_ratio: 0.3` or `0.7` (vs 0.5)
- **More paths**: `n_paths: 1000` (vs 100)

### 🎯 **Success metrics to improve:**
- Lower `hedge_std` (better risk control)
- Higher `hedge_sharpe` (better risk-adjusted returns)
- More stable option values

### 🚀 **Ready for next steps:**
- Add realized volatility features
- Implement proper deep hedging models
- Create backtesting framework